# 📔 Notebook Exercise: XGBoost Classifier Mastery

This notebook tests your understanding of the XGBoost training pipeline you built — from class-imbalance weighting and stratified cross-validation to cutoff threshold analysis and probability calibration.

Complete each exercise by replacing the `### YOUR CODE HERE ###` placeholders.  
Run the verification cells after each exercise to check your work.

---

## 🛠️ Setup & Mock Data Generation

Run this cell first. It generates a synthetic high-dimensional embedding dataset with realistic class imbalance (~1:6 spam-to-ham ratio), exactly matching what your real pipeline works with.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Synthetic 768-dim embeddings (mimics sentence-transformer output)
N_TRAIN, N_TEST, N_DIM = 800, 200, 768

# Class imbalance: ~14% spam — close to your real dataset ratio
y_train_full = np.random.choice([0, 1], size=N_TRAIN, p=[0.86, 0.14])
y_test_full  = np.random.choice([0, 1], size=N_TEST,  p=[0.86, 0.14])

# Spam embeddings cluster slightly differently (signal injected)
X_train_full = np.random.randn(N_TRAIN, N_DIM)
X_test_full  = np.random.randn(N_TEST,  N_DIM)
X_train_full[y_train_full == 1] += 0.35   # weak separable signal
X_test_full [y_test_full  == 1] += 0.35

print(f"Train shape: {X_train_full.shape}  |  Spam rate: {y_train_full.mean():.2%}")
print(f"Test  shape: {X_test_full.shape}   |  Spam rate: {y_test_full.mean():.2%}")


---

## 🏋️ Exercise 1: Class Imbalance Weight Calculation

### Instructions

Implement `compute_pos_weight`. It must:

1. Count the number of negative samples (label == 0) and positive samples (label == 1).
2. Return the ratio `neg / pos` — this is the `scale_pos_weight` XGBoost uses to re-balance gradient contributions.
3. Guard against division by zero using `max(pos, 1)`.

**Why this matters:** Without this correction, XGBoost minimises loss by predicting "ham" for everything — 86% accuracy, 0% recall. The weight steers the internal log-odds toward penalising missed spam more heavily.

In [ ]:
def compute_pos_weight(y):
    """
    Compute the positive-class weight for XGBoost's scale_pos_weight.
    Returns float: count(negatives) / count(positives).
    """
    ### YOUR CODE HERE (approx 3 lines) ###
    neg = None
    pos = None
    return None



### 🧪 Verification Test 1

In [ ]:
POS_WEIGHT = compute_pos_weight(y_train_full)
assert POS_WEIGHT is not None, "Return a value — got None"
assert isinstance(POS_WEIGHT, float), f"Expected float, got {type(POS_WEIGHT)}"
assert POS_WEIGHT > 1.0, "With ~14% spam the weight should be >> 1.0 (approximately 6–7x)"

# Edge-case: all-positive array must not crash
assert compute_pos_weight(np.ones(10, dtype=int)) == 0.0, "All-positive: neg/max(10,1) == 0.0"

print(f"✅ Exercise 1 Passed!  POS_WEIGHT = {POS_WEIGHT:.3f}")


---

## 🏋️ Exercise 2: Build the Optimised Parameter Dictionary

### Instructions

Assemble `build_xgb_params`. Return a `dict` that:

1. Sets the fixed base keys: `random_state=42`, `eval_metric='logloss'`, `tree_method='hist'`.
2. Sets the tree architecture: `n_estimators=500`, `max_depth=5`, `learning_rate=0.05`.
3. Sets stochastic sub-sampling: `subsample=0.85`, `colsample_bytree=0.85`.
4. Sets the regularisation knobs: `min_child_weight=3`, `reg_alpha=0.1` (L1), `reg_lambda=1.0` (L2).
5. Injects the computed `scale_pos_weight` from the argument passed in.

**Do not hard-code the weight** — accept it as the `pos_weight` parameter.

In [ ]:
def build_xgb_params(pos_weight):
    """
    Return the full XGBClassifier parameter dictionary.
    pos_weight : float — result from compute_pos_weight().
    """
    ### YOUR CODE HERE (approx 1 return statement) ###
    return {}



### 🧪 Verification Test 2

In [ ]:
params = build_xgb_params(POS_WEIGHT)

required_keys = [
    'random_state', 'eval_metric', 'tree_method',
    'n_estimators', 'max_depth', 'learning_rate',
    'subsample', 'colsample_bytree',
    'min_child_weight', 'reg_alpha', 'reg_lambda',
    'scale_pos_weight'
]
for k in required_keys:
    assert k in params, f"Missing key: '{k}'"

assert params['scale_pos_weight'] == POS_WEIGHT, "scale_pos_weight must use the passed-in pos_weight"
assert params['n_estimators'] == 500
assert params['reg_alpha'] == 0.1
assert params['reg_lambda'] == 1.0
print("✅ Exercise 2 Passed!  All parameter keys verified.")


---

## 🏋️ Exercise 3: Stratified Cross-Validation Engine

### Instructions

Implement `run_stratified_cv`. It must:

1. Create a `StratifiedKFold` with `n_splits=5, shuffle=True, random_state=42`.
2. For each fold: train a fresh `XGBClassifier(**params)`, predict **probabilities** on the validation split, apply `cutoff` to get binary predictions.
3. Collect per-fold weighted F1, spam precision, and spam recall into lists.
4. Return a dict with keys `mean_f1`, `mean_precision`, `mean_recall` (all floats).

**Key distinction vs plain `cross_val_score`:** by predicting `predict_proba` first and applying a custom threshold, you decouple model training from the decision boundary — which is exactly how your cutoff matrix analysis works.

In [ ]:
def run_stratified_cv(X, y, params, cutoff=0.50):
    """
    5-fold stratified CV returning averaged metrics at a given probability cutoff.
    """
    cv = None   ### YOUR CODE HERE: build StratifiedKFold ###

    f1s, precisions, recalls = [], [], []

    for train_idx, val_idx in cv.split(X, y):
        ### YOUR CODE HERE: slice X and y into train/val splits ###
        X_tr, X_val = None, None
        y_tr, y_val = None, None

        ### YOUR CODE HERE: instantiate, fit, predict_proba, apply cutoff ###
        model = None
        val_pred = None

        f1s.append(f1_score(y_val, val_pred, average='weighted'))
        precisions.append(precision_score(y_val, val_pred, pos_label=1, zero_division=0))
        recalls.append(recall_score(y_val, val_pred, pos_label=1, zero_division=0))

    ### YOUR CODE HERE: return dict with mean_f1, mean_precision, mean_recall ###
    return {}



### 🧪 Verification Test 3

In [ ]:
cv_results = run_stratified_cv(X_train_full, y_train_full, params)

assert set(cv_results.keys()) == {'mean_f1', 'mean_precision', 'mean_recall'}, \
    f"Unexpected keys: {cv_results.keys()}"
for k, v in cv_results.items():
    assert 0.0 <= v <= 1.0, f"{k} = {v} is out of [0, 1]"

print("✅ Exercise 3 Passed!")
print(f"   CV mean F1 (weighted) : {cv_results['mean_f1']:.4f}")
print(f"   CV mean spam precision: {cv_results['mean_precision']:.4f}")
print(f"   CV mean spam recall   : {cv_results['mean_recall']:.4f}")


---

## 🏋️ Exercise 4: Cutoff Threshold Analysis Matrix

### Instructions

Implement `cutoff_analysis`. Given fitted model probabilities `y_proba` and true labels `y_true`:

1. Sweep `cutoffs` from `0.10` to `0.95` in steps of `0.05` using `np.arange`.
2. For each cutoff compute: **spam precision**, **spam recall**, **false positives** (FP), **false negatives** (FN).
3. Return a `pd.DataFrame` with columns: `Cutoff`, `Spam Precision`, `Spam Recall`, `False Positives`, `False Negatives`.

**Reading the matrix:** Low cutoffs → high recall / low precision / many FPs. High cutoffs → high precision / low recall / many FNs. Your job is picking where the business cost of each error type balances.

In [ ]:
def cutoff_analysis(y_proba, y_true):
    """
    Sweep decision thresholds and return a precision-recall tradeoff DataFrame.
    """
    cutoffs = np.arange(0.10, 1.00, 0.05)
    rows = []

    for cutoff in cutoffs:
        ### YOUR CODE HERE: apply cutoff to get binary predictions ###
        y_pred_c = None

        ### YOUR CODE HERE: compute spam_precision, spam_recall, fp, fn ###
        spam_precision = None
        spam_recall    = None
        fp             = None
        fn             = None

        rows.append({
            'Cutoff': round(cutoff, 2),
            'Spam Precision': spam_precision,
            'Spam Recall': spam_recall,
            'False Positives': fp,
            'False Negatives': fn
        })

    ### YOUR CODE HERE: return pd.DataFrame(rows) ###
    return None



### 🧪 Verification Test 4

In [ ]:
# Train a model for the rest of the exercises
final_clf = xgb.XGBClassifier(**params)
final_clf.fit(X_train_full, y_train_full)
y_proba_test = final_clf.predict_proba(X_test_full)[:, 1]

results_df = cutoff_analysis(y_proba_test, y_test_full)

assert isinstance(results_df, pd.DataFrame), "Must return a DataFrame"
assert list(results_df.columns) == ['Cutoff', 'Spam Precision', 'Spam Recall', 'False Positives', 'False Negatives'], \
    f"Column mismatch: {list(results_df.columns)}"
assert len(results_df) == len(np.arange(0.10, 1.00, 0.05)), "Wrong number of cutoff rows"

# Monotonicity sanity: precision should generally rise as cutoff rises
low_prec  = results_df.loc[results_df['Cutoff'] == 0.10, 'Spam Precision'].values[0]
high_prec = results_df.loc[results_df['Cutoff'] == 0.90, 'Spam Precision'].values[0]
# (not strictly enforced — just display)

print("✅ Exercise 4 Passed!")
display(results_df)


---

## 🏋️ Exercise 5: Error Analysis — False Positive & False Negative Extraction

### Instructions

Implement `extract_errors`. Given a DataFrame `df` with a `text` column, `y_true` array, and `y_pred` array:

1. Add `y_true` and `y_pred` as columns to a copy of `df`.
2. Extract **false positives** — rows where `y_pred == 1` but `y_true == 0`.
3. Extract **false negatives** — rows where `y_pred == 0` but `y_true == 1`.
4. Return `(false_positives_df, false_negatives_df)` as a tuple.

**Why this matters:** FP/FN extraction is how you close the labelling loop — FNs go back to annotators as high-priority uncertain examples.

In [ ]:
def extract_errors(df, y_true, y_pred):
    """
    Returns (false_positives_df, false_negatives_df) from a predictions array.
    df must have the same row order as y_true / y_pred.
    """
    df = df.copy()

    ### YOUR CODE HERE: attach y_true and y_pred as columns ###

    ### YOUR CODE HERE: build false_positives and false_negatives masks ###
    false_positives = None
    false_negatives  = None

    return false_positives, false_negatives



### 🧪 Verification Test 5

In [ ]:
test_texts = pd.DataFrame({'text': [f'msg_{i}' for i in range(N_TEST)]})
y_pred_50 = (y_proba_test >= 0.5).astype(int)

fp_df, fn_df = extract_errors(test_texts, y_test_full, y_pred_50)

assert isinstance(fp_df, pd.DataFrame), "fp must be a DataFrame"
assert isinstance(fn_df, pd.DataFrame), "fn must be a DataFrame"

# FP rows: all predicted spam but actually ham
assert (fp_df['y_pred'] == 1).all(), "FP rows must all have y_pred == 1"
assert (fp_df['y_true'] == 0).all(), "FP rows must all have y_true == 0"

# FN rows: all predicted ham but actually spam
assert (fn_df['y_pred'] == 0).all(), "FN rows must all have y_pred == 0"
assert (fn_df['y_true'] == 1).all(), "FN rows must all have y_true == 1"

print(f"✅ Exercise 5 Passed!")
print(f"   False Positives : {len(fp_df)}")
print(f"   False Negatives : {len(fn_df)}")


---

## 🏋️ Exercise 6: Reliability (Calibration) Curve Plotter

### Instructions

Implement `plot_calibration_curve`. Given `y_true`, `y_proba`, and a `label`:

1. Use `calibration_curve(y_true, y_proba, n_bins=10)` to get `(prob_true, prob_pred)`.
2. Plot the calibration line with markers.
3. Plot the "perfectly calibrated" diagonal `[0,1] → [0,1]` as a dashed grey reference.
4. Add axis labels (`'Mean Predicted Probability'`, `'Fraction of Positives'`), a legend, grid, and title `f'Reliability Curve — {label}'`.

**Interpreting the curve:** A curve consistently above the diagonal means the model is under-confident (it says 0.3 but 50% of those cases are positive). Below means over-confident. XGBoost with `scale_pos_weight` often shifts the curve upward — this exercise lets you see exactly how much.

In [ ]:
def plot_calibration_curve(y_true, y_proba, label='Model'):
    """
    Plots a reliability diagram comparing predicted probability bins
    against the actual fraction of positives in each bin.
    """
    ### YOUR CODE HERE: call calibration_curve ###
    prob_true, prob_pred = None, None

    fig, ax = plt.subplots(figsize=(7, 5))

    ### YOUR CODE HERE: plot calibration line with markers ###

    ### YOUR CODE HERE: plot perfect calibration diagonal ###

    ### YOUR CODE HERE: labels, title, legend, grid ###

    plt.tight_layout()
    plt.show()



### 🧪 Verification Test 6

In [ ]:
# Visual check — should display a calibration plot without error
try:
    plot_calibration_curve(y_test_full, y_proba_test, label='XGBoost (baseline)')
    print("✅ Exercise 6 Passed!  Inspect the reliability curve above.")
except Exception as e:
    print(f"❌ Exercise 6 Failed: {e}")


---

## 🏋️ Exercise 7: scale_pos_weight Comparison Study

### Instructions

Implement `compare_pos_weights`. Given a list of weights to try `[1.0, 2.0, POS_WEIGHT]`:

1. For each weight: train a fresh `XGBClassifier` with that `scale_pos_weight` (keep all other params from `base_params`), predict probabilities, apply a 0.5 cutoff.
2. Compute: `Spam Precision`, `Spam Recall`, `Brier Score` (`brier_score_loss`).
3. Plot the calibration curve for each weight on the **same axes** (call `calibration_curve` directly).
4. Return a summary `pd.DataFrame` with columns: `POS_WEIGHT`, `Spam Precision`, `Spam Recall`, `Brier Score`.

**What to observe:** As you increase `scale_pos_weight`, the model shifts raw log-odds upward, effectively lowering the implicit decision boundary even at cutoff=0.5. Watch how this trades FP rate against FN rate.

In [ ]:
def compare_pos_weights(X_train, y_train, X_test, y_test, base_params, weights):
    """
    Trains one model per weight value and returns a comparison DataFrame.
    Also plots overlaid calibration curves.
    """
    import copy
    summary_rows = []
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Perfect')

    for pw in weights:
        ### YOUR CODE HERE: copy base_params, set scale_pos_weight = pw ###
        cur_params = None

        ### YOUR CODE HERE: train XGBClassifier, get y_proba, apply 0.5 cutoff ###
        y_pred_pw  = None
        y_proba_pw = None

        ### YOUR CODE HERE: compute spam_precision, spam_recall, brier ###
        spam_precision = None
        spam_recall    = None
        brier          = None

        ### YOUR CODE HERE: calibration_curve and ax.plot the line ###

        summary_rows.append({
            'POS_WEIGHT': round(pw, 3),
            'Spam Precision': spam_precision,
            'Spam Recall': spam_recall,
            'Brier Score': brier
        })

    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title('Reliability Curves — POS_WEIGHT Comparison')
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    return pd.DataFrame(summary_rows)



### 🧪 Verification Test 7

In [ ]:
summary_df = compare_pos_weights(
    X_train_full, y_train_full,
    X_test_full,  y_test_full,
    params,
    weights=[1.0, 2.0, POS_WEIGHT]
)

assert isinstance(summary_df, pd.DataFrame)
assert list(summary_df.columns) == ['POS_WEIGHT', 'Spam Precision', 'Spam Recall', 'Brier Score']
assert len(summary_df) == 3

print("✅ Exercise 7 Passed!")
display(summary_df)


---

## 💡 Hints (expand if you're stuck)

<details>
<summary><b>Exercise 1 — Pos-weight calculation</b></summary>

```python
neg = (y == 0).sum()
pos = (y == 1).sum()
return float(neg / max(pos, 1))
```
</details>

<details>
<summary><b>Exercise 2 — Parameter dictionary</b></summary>

```python
return dict(
    random_state=42, eval_metric='logloss', tree_method='hist',
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=pos_weight
)
```
</details>

<details>
<summary><b>Exercise 3 — Stratified CV engine</b></summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_tr, X_val = X[train_idx], X[val_idx]
y_tr, y_val = y[train_idx], y[val_idx]
model = xgb.XGBClassifier(**params)
model.fit(X_tr, y_tr, verbose=False)
val_pred = (model.predict_proba(X_val)[:, 1] >= cutoff).astype(int)
return {'mean_f1': np.mean(f1s), 'mean_precision': np.mean(precisions), 'mean_recall': np.mean(recalls)}
```
</details>

<details>
<summary><b>Exercise 4 — Cutoff matrix</b></summary>

```python
y_pred_c = (y_proba >= cutoff).astype(int)
spam_precision = precision_score(y_true, y_pred_c, pos_label=1, zero_division=0)
spam_recall    = recall_score(y_true, y_pred_c, pos_label=1, zero_division=0)
fp = ((y_pred_c == 1) & (y_true == 0)).sum()
fn = ((y_pred_c == 0) & (y_true == 1)).sum()
return pd.DataFrame(rows)
```
</details>

<details>
<summary><b>Exercise 5 — Error extraction</b></summary>

```python
df['y_true'] = y_true
df['y_pred'] = y_pred
false_positives = df[(df['y_pred'] == 1) & (df['y_true'] == 0)].copy()
false_negatives  = df[(df['y_pred'] == 0) & (df['y_true'] == 1)].copy()
```
</details>

<details>
<summary><b>Exercise 6 — Calibration curve</b></summary>

```python
prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=10)
ax.plot(prob_pred, prob_true, marker='o', label=label)
ax.plot([0,1],[0,1], linestyle='--', color='grey', label='Perfect')
```
</details>

<details>
<summary><b>Exercise 7 — POS_WEIGHT study</b></summary>

```python
cur_params = copy.deepcopy(base_params)
cur_params['scale_pos_weight'] = pw
clf_pw = xgb.XGBClassifier(**cur_params)
clf_pw.fit(X_train, y_train, verbose=False)
y_proba_pw = clf_pw.predict_proba(X_test)[:, 1]
y_pred_pw  = (y_proba_pw >= 0.5).astype(int)
prob_true, prob_pred = calibration_curve(y_test, y_proba_pw, n_bins=10)
ax.plot(prob_pred, prob_true, marker='o', label=f'POS_WEIGHT={pw:.2f}')
```
</details>
